<a href="https://colab.research.google.com/github/Dharmendra0305/PRODIGY_GA_01/blob/main/Text_Generation_with_GPT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 1: Environment Setup**

* Check GPU Availability



In [1]:
import torch
torch.cuda.is_available()

True

* Install Core Libraries

In [2]:
!pip install transformers datasets torch

* Verify Installation

In [3]:
import transformers
import datasets
import torch

print("Transformers Version:", transformers.__version__)
print("Datasets Version:", datasets.__version__)
print("PyTorch Version:", torch.__version__)

Transformers Version: 5.0.0
Datasets Version: 4.0.0
PyTorch Version: 2.10.0+cu128


# **Phase 2: Data Preparation**

* Collect Text

In [4]:
from google.colab import files

uploaded = files.upload()

Saving gpt.txt to gpt.txt


* Load Text

In [5]:
with open("gpt.txt", "r", encoding = "utf-8") as f:
  text = f.read()

print(text[:500])

﻿The Project Gutenberg eBook of The City of God, Volume I
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this e


* Tokenization

In [6]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Assign EOS token as pad token
tokenizer.pad_token = tokenizer.eos_token

# Tokenize text
tokens = tokenizer(text, return_tensors = "pt")

print(tokens["input_ids"][0][:50])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (336043 > 1024). Running this sequence through the model will result in indexing errors


tensor([  171,   119,   123,   464,  4935, 20336, 46566,   286,   383,  2254,
          286,  1793,    11, 14701,   314,   198,   220,   220,   220,   220,
          198,  1212, 46566,   318,   329,   262,   779,   286,  2687,  6609,
          287,   262,  1578,  1829,   290,   198,  1712,   584,  3354,   286,
          262,   995,   379,   645,  1575,   290,   351,  2048,   645,  8733])


* Chunking

In [7]:
block_size = 128

# Convert text into token IDs
token_ids = tokenizer.encode(text)

# Break into chunks
chunks = [token_ids[i:i + block_size] for i in range(0, len(token_ids), block_size)]

print("Number of chunks:", len(chunks))
print("First chunk:", chunks[0])

Number of chunks: 2626
First chunk: [171, 119, 123, 464, 4935, 20336, 46566, 286, 383, 2254, 286, 1793, 11, 14701, 314, 198, 220, 220, 220, 220, 198, 1212, 46566, 318, 329, 262, 779, 286, 2687, 6609, 287, 262, 1578, 1829, 290, 198, 1712, 584, 3354, 286, 262, 995, 379, 645, 1575, 290, 351, 2048, 645, 8733, 198, 10919, 15485, 13, 921, 743, 4866, 340, 11, 1577, 340, 1497, 393, 302, 12, 1904, 340, 739, 262, 2846, 198, 1659, 262, 4935, 20336, 13789, 3017, 351, 428, 46566, 393, 2691, 198, 265, 7324, 13, 70, 19028, 13, 2398, 13, 1002, 345, 389, 407, 5140, 287, 262, 1578, 1829, 11, 198, 5832, 481, 423, 284, 2198, 262, 3657, 286, 262, 1499, 810, 345, 389, 5140, 198, 19052, 1262, 428, 46566, 13, 198, 198, 19160, 25, 383, 2254]


In [8]:
from datasets import Dataset

dataset = Dataset.from_dict({"input_ids": chunks})
print(dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 2626
})


# **Phase 3: Fine-Tuning (Training)**



* Load the Base Model

In [9]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

* Prepare the Dataset

In [10]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer,
                                                mlm = False)           # causal LM (GPT-2) does not use masked language modeling

* Configure the Trainer

In [11]:
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [12]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",        # where to save checkpoints

    num_train_epochs = 3,            # train for 3 cycles

    per_device_train_batch_size = 2, # batch size per GPU

    save_steps = 500,                # save every 500 steps

    save_total_limit = 2,            # keep only last 2 checkpoints

    logging_steps = 100              # log every 100 steps
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = dataset,
    data_collator = data_collator
)

* Execute Training

In [13]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.613391
200,3.446253
300,3.468135
400,3.433010
500,3.415097
600,3.434052
700,3.369021
800,3.308755
900,3.278368
1000,3.338383


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3939, training_loss=3.060897399096599, metrics={'train_runtime': 540.9405, 'train_samples_per_second': 14.564, 'train_steps_per_second': 7.282, 'total_flos': 514614657024000.0, 'train_loss': 3.060897399096599, 'epoch': 3.0})

* Save the Model

In [14]:
trainer.save_model("./fine_tuned_gpt2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# **Phase 4: Generation & Evaluation**

* Load Fine-Tuned Model

In [15]:
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token         # Keep the padding fix

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

* Prepare a Prompt

In [16]:
prompt = "That empire was given to Rome not by the"

* Tokenize the Prompt

In [17]:
inputs = tokenizer(prompt, return_tensors = "pt", padding = True, return_attention_mask = True)

* Generate Text

In [18]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],

    pad_token_id = tokenizer.eos_token_id,  # explicitly set pad token

    max_length = 150,                       # total length of generated text

    num_return_sequences = 1,               # how many completions to generate

    temperature = 0.9,                      # creativity (lower = more focused, higher = more random)

    top_p = 0.85,                           # nucleus sampling (controls diversity)

    repetition_penalty = 1.2,               # discourages loops

    do_sample = True                        # enables sampling instead of greedy decoding
)

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

A model is trained on large and diverse masses of stars. These
and similar representations are commonly given to the gods, in order that
they may be believed either by those who worship them, or because they believe such things as I have
said concerning these goddesses; and it follows also from their testimony
that the images which men use, though not so accurately represented for all human eyesight (for some even our senses do see), can only become better
and clearer after a certain period of time, whilst we perceive them more clearly
by hearing them. For how could any one suppose that the gods whom he worships
should at once attain to celestial objects? But if this vision has been made intelligible
to all the world


# **Phase 5: Interactive Web App**

In [20]:
import gradio as gr

# 1. Load your trained model and tokenizer
model_path = "./custom-gpt2-final"
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# 2. Define the function that runs when a user clicks "Submit"
def generate_custom_text(prompt, temperature, max_length):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        **inputs,
        max_length=max_length,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# 3. Build the user interface
app = gr.Interface(
    fn=generate_custom_text,
    inputs=[
        gr.Textbox(lines=3, label="Starting Prompt", placeholder="Type here..."),
        gr.Slider(minimum=0.1, maximum=1.5, value=0.7, label="Temperature (Creativity)"),
        gr.Slider(minimum=10, maximum=200, step=10, value=50, label="Max Length")
    ],
    outputs=gr.Textbox(label="Generated Output"),
    title="Custom GPT-2 Text Generator"
)

# 4. Launch the web app
app.launch(share=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cff01d4b3bd2dccd8f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
